In [12]:
!pip install --upgrade pip

In [13]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@main git+https://github.com/unslothai/unsloth-zoo.git

In [ ]:
!pip install --upgrade pip
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -

In [14]:
!pip -q install -U trl

In [1]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [2]:
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
print(os.environ["UNSLOTH_RETURN_LOGITS"])

1


In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 500 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.nn.utils.rnn  import pad_sequence # Corrected import
import datasets
# # from utils import add_dataset_index, get_model_identifiers_from_yaml
# from data_module import convert_raw_data_to_model_format
import os
import pandas as pd
import numpy as np
import random
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, set_seed
import transformers
from transformers import Trainer
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [7]:
print(model.base_model)

LlamaModel(
  (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNor

In [8]:
# Let's identify the names of the modules, ensuring that we fine-tune the appropriate ones with adaptors.
[(n, type(m)) for n, m in model.named_modules()]

[('', transformers.models.llama.modeling_llama.LlamaForCausalLM),
 ('model', transformers.models.llama.modeling_llama.LlamaModel),
 ('model.embed_tokens', torch.nn.modules.sparse.Embedding),
 ('model.layers', torch.nn.modules.container.ModuleList),
 ('model.layers.0',
  transformers.models.llama.modeling_llama.LlamaDecoderLayer),
 ('model.layers.0.self_attn',
  transformers.models.llama.modeling_llama.LlamaAttention),
 ('model.layers.0.self_attn.q_proj', bitsandbytes.nn.modules.Linear4bit),
 ('model.layers.0.self_attn.k_proj', bitsandbytes.nn.modules.Linear4bit),
 ('model.layers.0.self_attn.v_proj', bitsandbytes.nn.modules.Linear4bit),
 ('model.layers.0.self_attn.o_proj', bitsandbytes.nn.modules.Linear4bit),
 ('model.layers.0.self_attn.rotary_emb',
  unsloth.models.llama.LlamaRotaryEmbedding),
 ('model.layers.0.mlp', transformers.models.llama.modeling_llama.LlamaMLP),
 ('model.layers.0.mlp.gate_proj', bitsandbytes.nn.modules.Linear4bit),
 ('model.layers.0.mlp.up_proj', bitsandbytes.nn.

In [9]:
alpaca_prompt = """Answer the following question:
### Question:
{}

### Answer:
{}"""

texts = []

def formatting_prompts_func(examples):
    texts = [alpaca_prompt.format(question, answer) for question, answer in zip(examples["question"], examples["answer"])]
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("locuslab/TOFU", "full", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.12.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [19]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 200,
        logging_steps = 10,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

In [ ]:
from datasets import load_dataset

alpaca_prompt = """Answer the following question:
### Question:
{}

### Answer:
{}"""

texts = []

def formatting_prompts_func(examples):
    texts = [alpaca_prompt.format(question, answer) for question, answer in zip(examples["question"], examples["answer"])]
    return { "text" : texts, }
pass

# Loading 'retain90' dataset
dataset_retain_90 = load_dataset("locuslab/TOFU", "retain90", split="train")
dataset_retain_90 = dataset_retain_90.map(formatting_prompts_func, batched=True)

In [ ]:
# Loading 'forget10' dataset
dataset_forget_10 = load_dataset("locuslab/TOFU", "forget10", split="train")
dataset_forget_10 = dataset_forget_10.map(formatting_prompts_func, batched=True)

In [ ]:
# Loading 'holdout01' dataset
dataset_holdout01 = load_dataset("locuslab/TOFU", "holdout01", split="train")
dataset_holdout01 = dataset_holdout01.map(formatting_prompts_func, batched=True)

In [ ]:
print(dataset_forget_10.column_names)

In [ ]:
print(dataset_forget_10[0])

In [ ]:
from datasets import concatenate_datasets

def add_factor_func(value: int):
    def _fn(examples):
        n = len(examples["text"])
        return {"factor": [value] * n}
    return _fn

forget_with_factor = (
    dataset_forget_10
    .map(formatting_prompts_func, batched=True)
    .map(add_factor_func(-1), batched=True)
)


retain_subset = dataset_retain_90.shuffle(seed=42).select(
    range(len(dataset_retain_90) // 9)
)

retain_with_factor = retain_subset.map(add_factor_func(1), batched=True)

unlearn_dataset = concatenate_datasets([forget_with_factor, retain_with_factor]).shuffle(seed=42)


In [ ]:
from collections import Counter

factors = [unlearn_dataset[i]["factor"] for i in range(len(unlearn_dataset))]
print(Counter(factors))


In [ ]:
unlearn_dataset = unlearn_dataset.shuffle(seed=42)

In [ ]:
# REPLACE YOUR tokenize_fn CELL WITH THIS:

def tokenize_fn(examples):
    # The template that separates Question from Answer
    # We use the exact string you likely have in your data
    response_template = "### Answer:\n"
    
    # 1. Encode the template to get its token IDs
    response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
    
    # 2. Encode the full text
    model_inputs = tokenizer(examples["text"], truncation=True, padding=False, max_length=512)
    
    labels = []
    for input_ids in model_inputs["input_ids"]:
        # Start with a purely masked label sequence
        lab = [-100] * len(input_ids)
        
        # Find the response template in the token sequence
        start_index = -1
        
        # Scan through input_ids to find the template sequence
        n = len(response_token_ids)
        for i in range(len(input_ids) - n):
            if input_ids[i:i+n] == response_token_ids:
                start_index = i + n  # The answer starts right AFTER the template
                break
        
        # If found, unmask everything after the template (the answer)
        if start_index != -1:
            for i in range(start_index, len(input_ids)):
                lab[i] = input_ids[i]
                
        labels.append(lab)
    
    model_inputs["labels"] = labels
    model_inputs["factor"] = examples["factor"]
    return model_inputs

In [ ]:
tokenized_unlearn = unlearn_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=unlearn_dataset.column_names,
)


In [ ]:
import math, random, torch
from torch.utils.data import Sampler
from transformers.trainer_utils import seed_worker

class DistributedMixedFactorBatchSampler(Sampler):
    """
    Produces batches with fixed composition each batch:
      k_forget examples where factor == -1
      k_retain examples where factor == +1
    Shards batches across distributed ranks so different GPUs see different batches.
    """
    def __init__(self, dataset, batch_size, k_forget, seed=0, drop_last=True,
                 rank=0, world_size=1):
        assert 0 < k_forget < batch_size, "Need both groups in each batch"
        self.dataset = dataset
        self.batch_size = batch_size
        self.kf = k_forget
        self.kr = batch_size - k_forget
        self.seed = seed
        self.drop_last = drop_last
        self.rank = rank
        self.world_size = world_size
        self.epoch = 0

        self.forget_idx = [i for i in range(len(dataset)) if int(dataset[i]["factor"]) == -1]
        self.retain_idx = [i for i in range(len(dataset)) if int(dataset[i]["factor"]) == 1]
        if len(self.forget_idx) == 0 or len(self.retain_idx) == 0:
            raise ValueError("Dataset must contain both factor=-1 and factor=+1 samples.")

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)

        f = self.forget_idx.copy()
        r = self.retain_idx.copy()
        rng.shuffle(f)
        rng.shuffle(r)

        def cycle(lst):
            while True:
                for x in lst:
                    yield x

        f_it = cycle(f)
        r_it = cycle(r)

        # Make enough global batches to cover the larger demand; smaller group cycles (oversamples).
        num_global_batches = math.ceil(max(len(f) / self.kf, len(r) / self.kr))

        # Shard by rank: each rank takes every world_size-th batch
        for batch_id in range(num_global_batches):
            if (batch_id % self.world_size) != self.rank:
                # Skip batches not assigned to this rank
                # Still advance iterators to keep all ranks in sync? Not necessary since iterators are local.
                # But we must still "consume" the same amount to maintain distribution? Not required.
                continue

            batch = [next(f_it) for _ in range(self.kf)] + [next(r_it) for _ in range(self.kr)]
            rng.shuffle(batch)
            yield batch

    def __len__(self):
        # approximate per-rank batches
        num_global_batches = math.ceil(len(self.dataset) / self.batch_size)
        return math.ceil(num_global_batches / self.world_size)


In [ ]:
import torch
from transformers import DataCollatorWithPadding

base_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

class CollatorWithFactorPadLabels:
    def __init__(self, base):
        self.base = base

    def __call__(self, features):
        factors = torch.tensor([f["factor"] for f in features], dtype=torch.long)

        # separate labels so we can pad them with -100
        labels = [f["labels"] for f in features]
        feats_wo = [{k: v for k, v in f.items() if k not in ["factor", "labels"]} for f in features]

        batch = self.base(feats_wo)
        max_len = batch["input_ids"].shape[1]

        padded_labels = []
        for lab in labels:
            lab = lab[:max_len]
            padded_labels.append(lab + [-100] * (max_len - len(lab)))

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long, device=batch["input_ids"].device)
        batch["factor"] = factors.to(batch["input_ids"].device)

        return batch

data_collator = CollatorWithFactorPadLabels(base_collator)


In [ ]:
class GAFTSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        factors = inputs.pop("factor")  # (bs,)
        f_mask = factors == -1
        r_mask = factors == 1

        def subset(mask):
            if not mask.any():
                return None
            out = {}
            for k, v in inputs.items():
                if isinstance(v, torch.Tensor):
                    out[k] = v[mask]
            return out

        forget_inputs = subset(f_mask)
        retain_inputs = subset(r_mask)

        loss = 0.0
        out_f = out_r = None

        if forget_inputs is not None:
            out_f = model(**forget_inputs)
            loss = loss - out_f.loss  # gradient ascent on forget

        if retain_inputs is not None:
            out_r = model(**retain_inputs)
            loss = loss + (self.gamma * out_r.loss)

        outputs = (out_f, out_r)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        train_dataset = self.train_dataset
        if train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")

        bs = self.args.per_device_train_batch_size
        if bs < 2:
            raise ValueError("Need batch_size >= 2 to enforce mixed batches.")

        world_size = dist.get_world_size() if dist.is_available() and dist.is_initialized() else 1
        rank = dist.get_rank() if dist.is_available() and dist.is_initialized() else 0

        sampler = DistributedMixedFactorBatchSampler(
            dataset=train_dataset,
            batch_size=bs,
            k_forget=1,  # for bs=2 -> 1 forget + 1 retain
            seed=self.args.seed,
            drop_last=True,
            rank=rank,
            world_size=world_size,
        )
        sampler.set_epoch(int(self.state.epoch or 0))

        return DataLoader(
            train_dataset,
            batch_sampler=sampler,
            collate_fn=self.data_collator,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
            worker_init_fn=seed_worker,
        )

In [ ]:
from trl import SFTConfig


# 3) Instantiate trainer (note gamma added)
trainer = GAFTSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",              # if using TRL SFT datasets; otherwise omit
    max_seq_length=max_seq_length,
    data_collator=data_collator,      # <-- important
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,      # must be >=2 for mixed forget/retain
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=200,
        logging_steps=10,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
    gamma=1.0,  # <--- add this kwarg in your __init__
)


In [ ]:
trainer.train()